In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/annotated/checkpoints/post_cell_1.pickle

trying: ['orig_output']
me:  2
trying: ['pd']
me:  0
trying: ['load_lineitem']
me:  1
trying: ['STORAGE_OPTS']
me:  0
trying: ['DATA_ROOT']
me:  0
trying: ['load_supplier']
me:  3
trying: ['lineitem']


me:  1
trying: ['supplier']
me:  3
trying: ['tpch_parent']
me:  0


Declaring variable pd
Declaring variable STORAGE_OPTS
Declaring variable DATA_ROOT
Declaring variable tpch_parent
Declaring variable load_lineitem
Declaring variable lineitem
Declaring variable orig_output
Declaring variable load_supplier
Declaring variable supplier


In [4]:
%%RecordEvent
%%time
### cell 2 ###

# Optimized GPU workflow using cudf.pandas
# Define date bounds as strings to push comparisons to the GPU
date_start = "1996-01-01"
date_end = "1996-04-01"

# Filter, compute revenue parts, aggregate and pick top supplier in one pipeline
top_revenue = (
    lineitem
    .loc[
        (lineitem.L_SHIPDATE >= date_start) & (lineitem.L_SHIPDATE < date_end),
        ["L_SUPPKEY", "L_EXTENDEDPRICE", "L_DISCOUNT"]
    ]
    .assign(REVENUE_PARTS=lambda df: df.L_EXTENDEDPRICE * (1 - df.L_DISCOUNT))
    .groupby("L_SUPPKEY")["REVENUE_PARTS"].sum()
    .reset_index()
    .nlargest(1, "REVENUE_PARTS")
    .rename(columns={"L_SUPPKEY": "S_SUPPKEY", "REVENUE_PARTS": "TOTAL_REVENUE"})
)

# Join with supplier details and select final columns
total = (
    top_revenue
    .merge(
        supplier[["S_SUPPKEY", "S_NAME", "S_ADDRESS", "S_PHONE"]],
        on="S_SUPPKEY"
    )
    [["S_SUPPKEY", "S_NAME", "S_ADDRESS", "S_PHONE", "TOTAL_REVENUE"]]
)


CPU times: user 75.2 ms, sys: 19.7 ms, total: 94.9 ms
Wall time: 102 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/rewritten/o4_mini_high/checkpoints/post_cell_2_try_1.pickle

migration speed (bps): 796847969.0600156
---------------------------
variables to migrate:
pd 72
date_start 59
orig_output 16
STORAGE_OPTS 64
date_end 59
tpch_parent 54
supplier 48
load_lineitem 144
top_revenue 48
lineitem 48
DATA_ROOT 80
load_supplier 144
total 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/rewritten/o4_mini_high/checkpoints/post_cell_2_try_1.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['supplier']
Future variables ['lineitem']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['total', 'top_revenue', 'date_start', 'date_end']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/opt_cell_exec_info_2_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[2], f)


In [8]:
opt_output = Out.get(4)

In [9]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/annotated/checkpoints/post_cell_2.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['lineitem']


me:  1
trying: ['load_supplier']
me:  3
trying: ['lineitem_filtered']
me:  5
trying: ['supplier']
me:  3
trying: ['max_revenue']
me:  5
trying: ['orig_output']
me:  2
trying: ['STORAGE_OPTS']
me:  0
trying: ['tpch_parent']
me:  0
trying: ['DATA_ROOT']
me:  0
trying: ['total']
me:  5
trying: ['pd']
me:  0
trying: ['supplier_filtered']
me:  5
trying: ['load_lineitem']
me:  1
trying: ['revenue_table']
me:  5


Declaring variable STORAGE_OPTS
Declaring variable tpch_parent
Declaring variable DATA_ROOT
Declaring variable pd
Declaring variable lineitem
Declaring variable load_lineitem
Declaring variable orig_output
Declaring variable load_supplier
Declaring variable supplier
Declaring variable lineitem_filtered
Declaring variable max_revenue
Declaring variable total
Declaring variable supplier_filtered
Declaring variable revenue_table
